# Einops

[https://einops.rocks/1-einops-basics/](https://einops.rocks/1-einops-basics/)

[https://einops.rocks/2-einops-for-deep-learning/](https://einops.rocks/2-einops-for-deep-learning/)

`rearrange` doesn't change number of elements and covers different numpy functions (like `transpose`, `reshape`, `stack`, `concatenate`, `squeeze` and `expand_dims`)

`reduce` combines same reordering syntax with reductions (`mean`, `min`, `max`, `sum`, `prod`, and any others)

`repeat` additionally covers repeating and tiling

composition and decomposition of axes are a corner stone, they can and should be used together



In [2]:
import torch
from einops import rearrange, repeat, reduce

In [ ]:
from einops.layers.torch import Reduce, Rearrange #can be used in pytorch networks and be used for the backprop

In [17]:
def get_data():
    data = torch.zeros(size=(1,2,3,4))
    for i in range(data.shape[-1]):
        data[:,:,:,i] = i
    return data

## rearrange

In [18]:
data = get_data()

In [19]:
data.permute(0,2,3,1)

tensor([[[[0., 0.],
          [1., 1.],
          [2., 2.],
          [3., 3.]],

         [[0., 0.],
          [1., 1.],
          [2., 2.],
          [3., 3.]],

         [[0., 0.],
          [1., 1.],
          [2., 2.],
          [3., 3.]]]])

In [20]:
rearrange(data, "a b c d -> a c d b")

tensor([[[[0., 0.],
          [1., 1.],
          [2., 2.],
          [3., 3.]],

         [[0., 0.],
          [1., 1.],
          [2., 2.],
          [3., 3.]],

         [[0., 0.],
          [1., 1.],
          [2., 2.],
          [3., 3.]]]])

In [21]:
rearrange(data, "b h w c -> (b h) w c")

tensor([[[0., 1., 2., 3.],
         [0., 1., 2., 3.],
         [0., 1., 2., 3.]],

        [[0., 1., 2., 3.],
         [0., 1., 2., 3.],
         [0., 1., 2., 3.]]])

In [22]:
rearrange(data, "m n o p -> m (n o) p")

tensor([[[0., 1., 2., 3.],
         [0., 1., 2., 3.],
         [0., 1., 2., 3.],
         [0., 1., 2., 3.],
         [0., 1., 2., 3.],
         [0., 1., 2., 3.]]])

In [23]:
rearrange(data, "a b c (d e) -> a b c d e", e=2)

tensor([[[[[0., 1.],
           [2., 3.]],

          [[0., 1.],
           [2., 3.]],

          [[0., 1.],
           [2., 3.]]],


         [[[0., 1.],
           [2., 3.]],

          [[0., 1.],
           [2., 3.]],

          [[0., 1.],
           [2., 3.]]]]])

In [24]:
data.data_ptr(), data.is_contiguous()

(40303802240, True)

In [25]:
rearrange(data, "a b c d -> a b d c").is_contiguous() #data is changed in the memory

False

In [26]:
rearrange(data, "a b c d -> a b (c d)").is_contiguous()

True

In [27]:
tdata = rearrange(data, "a b c d -> a b d c")
data.data_ptr(), tdata.data_ptr()

(40303802240, 40303802240)

In [117]:
rearrange(data, "a b c d -> 1 a b c d 1").shape

torch.Size([1, 1, 2, 3, 4, 1])

## mean

In [29]:
data = get_data()
data.shape

torch.Size([1, 2, 3, 4])

In [31]:
data.mean(dim=1).shape

torch.Size([1, 3, 4])

In [33]:
reduce(data, "a b c d -> a c d", "mean").shape

torch.Size([1, 3, 4])

In [34]:
torch.all(data.mean(dim=1) == reduce(data, "a b c d -> a c d", "mean"))

tensor(True)

In [58]:
data = torch.zeros(size=(2,6,6,3))
for i in range(12):
    data[i//6][i%6] = i
data[0][0] = 11 #to check if the broadcasting works and min is really compared based on this
data[1][0][0][0] = -10

In [65]:
data.min(dim=0).values.shape, data.min(dim=0).values[0][0]

(torch.Size([6, 6, 3]), tensor([-10.,   6.,   6.]))

In [66]:
reduce(data, "a b c d -> b c d", "min").shape, reduce(data, "a b c d -> b c d", "min")[0][0]

(torch.Size([6, 6, 3]), tensor([-10.,   6.,   6.]))

In [105]:
data = torch.zeros(size=(2,3,6,6))
count = 0.0
for i in range(data.shape[0]):
    for j in range(data.shape[1]):
        for k in range(data.shape[2]):
            for l in range(data.shape[3]):
                data[i,j,k,l] = count
                count += 1

In [ ]:
data = torch.randn(size=(2,3,6,6))
reduced_data = reduce(data, "n c (h1 h2) (w1 w2) -> n c h1 w1", "mean", h2=2, w2=2)
torch.allclose(torch.nn.AvgPool2d(kernel_size=2)(data), reduced_data, ), reduced_data.shape

(True, torch.Size([2, 3, 3, 3]))

In [ ]:
data = torch.ones(4,4)
reduce(data, "w h -> () w h", "min").shape #like unsqueeze

torch.Size([1, 4, 4])

## repeat

reduce and repeat are like opposite of each other: first one reduces amount of elements, second one increases.



In [120]:
data = torch.randn(size=(2,3))
data

tensor([[ 0.6398,  0.3187, -1.0419],
        [-1.0648,  1.0076,  0.6749]])

In [123]:
repeat(data, "a b -> a c b", c=2)

tensor([[[ 0.6398,  0.3187, -1.0419],
         [ 0.6398,  0.3187, -1.0419]],

        [[-1.0648,  1.0076,  0.6749],
         [-1.0648,  1.0076,  0.6749]]])

In [129]:
repeated = repeat(data, "a b -> a c b", c=2)
reduced = reduce(repeated, "a c b -> a b", "min")
torch.all(data == reduced) # data -> repeated -> reduced

tensor(True)